In [81]:
# baseline_tier_three.ipynb
# Tier 3 Baseline: Spatio-Temporal Graph Attention Networks (GAT-LSTM)

import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score



In [82]:
# --- 1. DATA LOADING & PREPROCESSING ---
def load_and_preprocess(file_path='C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/final_dataset.parquet'):
    df = pd.read_parquet(file_path)
    # print(df.columns)
    # Ensure time_id exists and data is sorted for temporal consistency

    # Create the 'Remoteness' proxy identified in your EDA
    if 'source_Dist_Highway_Entrance' in df.columns:
            df['source_Remoteness_Index'] = (df['source_Dist_Highway_Entrance'] + 
                                    df['source_Dist_Major_Transfer_Station']) / 2
            
            df['target_Remoteness_Index'] = (df['target_Dist_Highway_Entrance'] + 
                                    df['target_Dist_Major_Transfer_Station']) / 2

    if 'time_id' not in df.columns:
        # If time_id is missing, we create one based on the index or timestamp
        df['time_id'] = df.groupby('Date').cumcount() 

    df= df.fillna(0)
    
    return df.sort_values(['time_id', 'source']) 

In [83]:
# 1. GRAPH CONSTRUCTION LOGIC
def build_graph_data(df, node_features, edge_index, target_col):
    """
    Converts tabular data into a list of PyTorch Geometric Data objects.
    Each object represents the state of the whole network at one point in time.
    """
    graphs = []
    time_steps = df['time_id'].unique() # Assuming your EDA has a time_id
    
    for t in time_steps:
        snap = df[df['time_id'] == t]
        
        # Node features: [Num_Stations, Num_Features]
        x = torch.tensor(snap[node_features].values, dtype=torch.float32)
        
        # Target: [Num_Stations] (Delay classification)
        y = torch.tensor(snap[target_col].values, dtype=torch.float32)
        
        # Graph snapshot
        data = Data(x=x, edge_index=edge_index, y=y)
        graphs.append(data)
    return graphs



In [84]:
# 2. TIER 3 ARCHITECTURE: GAT-LSTM CELL
class GATLSTMCell(nn.Module):
    def __init__(self, in_channels, out_channels, heads=2):
        super(GATLSTMCell, self).__init__()
        # Spatial Layer: Graph Attention
        self.gat = GATv2Conv(in_channels, out_channels // heads, heads=heads)
        
        # Temporal Layer: LSTM Cell
        self.lstm = nn.LSTMCell(out_channels, out_channels)
        
    def forward(self, x, edge_index, h, c):
        # 1. Spatial aggregation
        spatial_out = self.gat(x, edge_index)
        # 2. Temporal update
        h_next, c_next = self.lstm(spatial_out, (h, c))
        return h_next, c_next



In [85]:
# 3. FULL MODEL
class SpatioTemporalModel(nn.Module):
    def __init__(self, node_in_channels, hidden_channels):
        super().__init__()
        self.st_cell = GATLSTMCell(node_in_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, 1)
        self.hidden_channels = hidden_channels

    def forward(self, sequence, edge_index):
        # sequence: List of Data objects (T time steps)
        batch_size = sequence[0].x.size(0)
        h = torch.zeros(batch_size, self.hidden_channels).to(sequence[0].x.device)
        c = torch.zeros(batch_size, self.hidden_channels).to(sequence[0].x.device)
        
        # Pass through the sequence (Temporal evolution)
        for snapshot in sequence:
            h, c = self.st_cell(snapshot.x, edge_index, h, c)
        
        return self.classifier(h)



In [ ]:
# --- Configuration ---
FILE_PATH = 'C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/final_dataset.parquet'
SUBSET_FRACTION = 0.4  # Change to 1.0 for full run
EPOCHS = 10
HIDDEN_DIM = 32

# Load data
full_df = load_and_preprocess(FILE_PATH)

# Define Features
operational_feats = ['Rides planned', 'DR', 'RH', 'SQ', 'TG', 'TN', 'TX', 'RHX']
socio_feats = ['target_SES_Score_Wealth_Avg', 'target_SES_Score_Education_Avg', 'target_TotalVandalism', 'target_Remoteness_Index']
target = 'Disrupted'

# Static Edge Index (Example: you would derive this from your railway adjacency matrix)
# This must match the number of unique stations in your df snapshots
num_nodes = full_df['source'].nunique()
# Dummy edges for script structure (Connect node i to i+1)
edge_index = torch.tensor([list(range(num_nodes-1)) + list(range(1, num_nodes)),
                            list(range(1, num_nodes)) + list(range(num_nodes-1))], dtype=torch.long)

# Comparison Loop
for name, features in [("Operational GNN", operational_feats), 
                        ("Socio-Spatio-Temporal GNN", operational_feats + socio_feats)]:
    
    print(f"\n>>> Starting: {name}")
    
    # 1. Subset Data
    if SUBSET_FRACTION < 1.0:
        times = full_df['time_id'].unique()
        subset_times = times[:int(len(times) * SUBSET_FRACTION)]
        df_work = full_df[full_df['time_id'].isin(subset_times)].copy()
    else:
        df_work = full_df.copy()

    # 2. Scaling
    scaler = StandardScaler()
    df_work[features] = scaler.fit_transform(df_work[features])
    
    # 3. Build Snapshots
    graph_sequence = build_graph_data(df_work, features, edge_index, target)
    
    # 4. Init Model
    model = SpatioTemporalModel(len(features), HIDDEN_DIM)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.BCEWithLogitsLoss()
    
    # 5. Training
    model.train()
    for epoch in range(EPOCHS):
        optimizer.zero_grad()
        # Predict based on the sequence
        logits = model(graph_sequence, edge_index)
        # Target is the disruption state at the final time step
        y_true = graph_sequence[-1].y.view(-1, 1)

        num_nodes = y_true.size(0)
        num_pos = y_true.sum()
        num_neg = num_nodes - num_pos
        pos_weight = num_neg / (num_pos + 1e-5) # avoid div by zero

        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))

        
        loss = criterion(logits, y_true)
        loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        if (epoch + 1) % 2 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f}")

    # 6. Basic Evaluation
    model.eval()
    with torch.no_grad():
        preds = torch.sigmoid(model(graph_sequence, edge_index)).numpy()
        actuals = graph_sequence[-1].y.numpy()
        binary_preds = (preds > 0.5).astype(int)
        acc = balanced_accuracy_score(actuals, binary_preds)
        print(f"Result for {name}: Balanced Accuracy = {acc:.4f}")


>>> Starting: Operational GNN
Epoch 2/10 | Loss: 0.5802
Epoch 4/10 | Loss: 0.4960
Epoch 6/10 | Loss: 0.3994
Epoch 8/10 | Loss: 0.2867
Epoch 10/10 | Loss: 0.1879
Result for Operational GNN: Balanced Accuracy = 0.5000

>>> Starting: Socio-Spatio-Temporal GNN
Epoch 2/10 | Loss: 0.6020
Epoch 4/10 | Loss: 0.5014
Epoch 6/10 | Loss: 0.3799
Epoch 8/10 | Loss: 0.2473
Epoch 10/10 | Loss: 0.1491
Result for Socio-Spatio-Temporal GNN: Balanced Accuracy = 0.5000
